# Финал: обучение на всей истории, модели для контейнера и сабмит

Правило перевода количества в целое после этого прогона поменяли: соответствие прогноз -> целое (05_intmap) на `recent` завышало объём, и в финале осталось целое под SMAPE по Пуассону. Модели те же, поэтому `model/config.json` и `submission/submission.csv` пересчитаны из сохранённых моделей локально. Выводы ниже с прогона, где ещё было соответствие.

Итоговый конфиг - `src/final.py` (что прошло проверку на трёх фолдах; журнал экспериментов - `results/ideas.csv`, ноутбуки `05_*`). Модели учатся на всей истории по 2026-07-26 тем же `Forecaster` (`src/forecaster.py`), что работает в контейнере, и сохраняются в `model/`: CatBoost - `.cbm`, LightGBM - `.txt`, конфиг (признаки, порог, соответствие "прогноз -> целое", состав ансамбля, метрики бэктеста, версии библиотек) - `config.json`.

Сабмит считается **из сохранённых моделей**, как в контейнере. Проверка: прогноз по последним 180 дням (столько приходит в контейнер) совпадает с прогнозом по всей истории - признаки считаются по окну не длиннее 24 недель.

На Kaggle: `make run NB=06_final` (или `push` + `pull`) - модели и `submission.csv` приходят в `outputs/kaggle/06_final/`.

In [1]:
import sys; sys.path.insert(0, "../src")   # код проекта - в src/

import numpy as np
import pandas as pd
from core import *
import classifier, regressor
import final
from forecaster import Forecaster, library_versions

classifier.ORIGIN_STEP = regressor.ORIGIN_STEP = final.ORIGIN_STEP   # как в валидации: каждая 2-я неделя отсчёта
sales, stock = load_data()
print("данные по", sales.date.max().date(), "| ансамбль:", final.PARAMS["ensemble"], "| порог:", final.PARAMS["threshold"])
library_versions()

данные по 2026-07-26 | ансамбль: [{}, {'random_seed': 7}, {'library': 'lightgbm'}] | порог: (0.55, 0.5)


{'python': '3.13.15',
 'pandas': '2.3.3',
 'numpy': '2.1.3',
 'scipy': '1.16.3',
 'catboost': '1.2.10',
 'lightgbm': '4.6.0',
 'holidays': '0.103'}

## Обучение и сохранение

In [2]:
MODEL_DIR = OUT_DIR / "model"
forecaster = Forecaster.fit(sales, stock, **final.PARAMS)
version = forecaster.save(MODEL_DIR, backtest=final.BACKTEST)
print("версия модели:", version, "|", sorted(p.name for p in MODEL_DIR.iterdir()))

версия модели: d2fdfa3ba3aa | ['clf_0.cbm', 'clf_1.cbm', 'clf_2.txt', 'config.json', 'reg_0.cbm', 'reg_1.cbm', 'reg_2.txt']


## Сабмит из сохранённых моделей; прогноз по 180 дням = по всей истории

In [3]:
loaded = Forecaster.load(MODEL_DIR)
sub = loaded.predict(sales, stock)
start = sales.date.max() - pd.Timedelta(days=179)                      # 180 дней - как в контейнере
sub180 = loaded.predict(sales[sales.date >= start], stock[stock.date >= start])
key = ["week", "warehouse_name", "article"]
both = sub.merge(sub180, on=key, how="left", suffixes=("", "_180"))
print("строк: по всей истории", len(sub), "| по 180 дням", len(sub180),
      "| совпадает значений:", int((both.quantity == both.quantity_180).sum()), "из", len(both),
      "| пар нет в 180 днях:", int(both.quantity_180.isna().sum() // 9),
      "| у них прогноз:", both.loc[both.quantity_180.isna(), "quantity"].sum())
print(check_submission(sub) or "формат ок")
sub.to_csv(SUBMISSION_CSV, index=False, date_format="%Y-%m-%d")
print(len(sub), "строк, недели", sub.week.min().date(), "...", sub.week.max().date(),
      "| штук за 9 недель:", round(sub.quantity.sum()), "| доля нулей:", round((sub.quantity == 0).mean(), 3),
      "| версия модели:", loaded.version)

строк: по всей истории 79101 | по 180 дням 79101 | совпадает значений: 79101 из 79101 | пар нет в 180 днях: 0 | у них прогноз: 0.0


формат ок


79101 строк, недели 2026-07-27 ... 2026-09-21 | штук за 9 недель: 234778 | доля нулей: 0.549 | версия модели: d2fdfa3ba3aa
